## F1 RACE FINISHING POSITION PREDICTOR

### Goal

We want to predict the finishing position of race drivers in a Formula 1 grand prix, based on historical performances of the drivers/ teams, characteristics of the tracks, and race specific (FP performance, qualifying, weather etc) and season specific (team and driver momentum) details. 

#### Challenges

This problem is hard because we need a model that is : 
- Robust to outliers (DNFs, safety cars, weather, team strategies, etc)
- Deals well with tabular, mixed type features.
- Captures Non linear interactions (fast track + slippery conditions + low overtaking ability)

#### **XGBoost**
We choose XGboost over other models because, linear models cannot fit non linear interactions, random forests are a good baseline, but still lag behind Boosting as boosting predicts residuals with new instead of averaging down new trees. Neural networks need a lot more data, gradient boosting doesn't internally provide regularization (can over fit to outliers easily) . Other boosting models like LightGBM and CatBoost are good alternatives, and is the planned second iteration. [An easy benefit of XGboost is the native handling of missing values]. 

#### Success Metrics

We evaluate the model on the held out set of recent races using:

  - MAE on finishing position 
        Target: < 3.0 positions on average
  
  - Top-3 (podium) hit rate
        Target : > 60% of actual podium finishers in our predicted top 3. 
  
  - Top-10 (points) hit rate 
        Target: > 75% 

  - Spearman rank correlation per race
        Target: > 0.65 averaged across test races.
        Rationale: To represent models ability to get the order correct.

#### Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

import xgboost as xgb

We do not use Sickit learn's `GradientBoostingRegressor` because it is materially slower. 

### Configuration

We centralize the configuration - meaning hyperparameter settings can all be tracked and set in one place. 

In [ ]:
CONFIG = {
    # path to data file
    "data_path": "data/f1_model_data.parquet",

    # training seasons
    "train_seasons": list(range(2014, 2024)),
    # test seasons
    "test_seasons": [2024],

    # --Hyperparameters for XGBoost--

    "xgb_params": {

        # Loss function for residuals. Matches our MAE evaluation metric. 
        "objective": "reg:squarederror",

        # Number of trees in the ensemble. We Set-high, and rely on early-stopping to pick the best iteration
        "n_estimators":2000,

        # Max depth each tree should grow. F1 maybe has ~30-50 useful features, depth 6 lets the model
        # learn interactions without memorizing. Depth >8 on this size overfits quickly. 
        "max_depth": 6,

        # Learning rate - small LR + many trees is standard. Too big underfits the residual xgb structure.
        "learning_rate": 0.05,

        # Row subsampling helps in regularization. Each tree sees 80% of rows which decorrelates trees and reduces overfitting..
        "subsample": 0.8,

        # column subsampling helps in regularization, reduces overfitting to certain features. Each tree sees 80% of columns.
        "colsample_bytree": 0.8,

        # L2 regularization term on weights. Discourages any single leaf from dominating. 
        "reg_lambda": 1.0, 

        # Min. Child weight - minimum sum of hessians (in our case, hessians for all rows is 1, so it implies the min number of rows in the leaf)
        "min_child_weight": 1,

        # without this every run would be different.Allows reproducibility of results.
        "random_state": 42,   

        # Typically 5-10x faster than exact algo without loss to accuracy.
        "tree_method": "hist", 

        # Suppress per-iteration warnings
        "verbosity": 1,

    }, 


# --- Cross-Validation ---

# We group by 'race_id' so a single race doesn't get split up between train/test folds. This is important to prevent data about the race from leaking into training.

"cv_folds":5,

# Early stopping rounds - stop if validation MAE doesn't improve for this many iterations. 

"early_stopping_rounds": 50,

}

### Data Loading

Simple function to lead the data parquet file into a dataframe. 

In [ ]:
def load_data(path:str) -> pd.DataFrame:
    df = pd.read_parquet(path)

    # Optional : Sanity checks - 
    assert df["Position"].between(1, 21).all()
    assert df["race_id"].notna().all()

    return df

##### **Split Features - Target**

In [ ]:
def split_features_target(df: pd.DataFrame):

    """
    Separate the dataframe into:
    X  — feature matrix
    y  — target vector (finishing_position)
    groups — race_id, used for group-aware CV
    """

    target_col = "Position"

    # We drop identifiers to risk overfitting to certain drivers/ tracks, and rely on
    # our features to capture the relevant information about the driver and the track.

    drop_cols = ["Position", "Season", "Round", "race_id",
             "Abbreviation", "TeamName", "Location", "Circuit"]

    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols].copy()
    y = df[target_col].copy()
    groups = df["race_id"].copy()

    # For now we do not have any categoricaL features, but if we decide to add them - 
    
    # Convert object/string columns to categorical for XGBoost native handling.
    for col in X.select_dtypes(include=["object"]).columns:
        X[col] = X[col].astype("category")

### Training   

In [ ]:
def train_with_cv(X: pd.DataFrame, y: pd.Series, groups: pd.Series, config: dict):

    """ 
    Train XGBoost using GroupKFold cross-validation

    Returns a list of trained models (one per fold). Ensembling fold models at inference time gives an MAE improvement.
    Final predictions are the mean of five GroupKFold models; averaging decorrelated fold models reduces variance.
    """

    cv = GroupKFold(n_splits=config["cv_folds"])
    fold_models = []
    fold_metrics = []

    # The for loop below emulates a cross validation style of training models - splitting rows into train and validation sets, each w